In [0]:
-- =============================================================
-- 1. GS_CC_Members
-- =============================================================
CREATE OR REPLACE TABLE data_exploration_hub.marketing_hub.GS_CC_Members AS
WITH latest_snapshot AS (
  SELECT MAX(snapshot_date_id) AS snapshot_date_id
  FROM   cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
)
SELECT m.cde_id,
       m.cc_user_profile_id,
       CASE m.plan_id
         WHEN 'monthly1'  THEN 'Monthly'
         WHEN 'annual1'   THEN 'Annual'
         WHEN 'gift-plan' THEN 'Gift Plan'
         ELSE m.plan_id
       END                                                          AS plan_id,
       m.renewal_status,
       m.recognitions_redeemed,
       m.recognitions_left,
       TO_DATE(m.current_term_start_datetime)                       AS current_term_start_datetime,
       TO_DATE(m.current_term_end_datetime)                         AS current_term_end_datetime,
       TO_DATE(CAST(m.snapshot_date_id AS STRING), 'yyyyMMdd')      AS snapshot_date_id
FROM   cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot m
JOIN   latest_snapshot ls ON m.snapshot_date_id = ls.snapshot_date_id
WHERE  m.recognitions_redeemed > 0;

-- =============================================================
-- 2. GS_CC_Transactions
-- =============================================================
CREATE OR REPLACE TABLE data_exploration_hub.marketing_hub.GS_CC_Transactions AS
WITH members AS (
  SELECT DISTINCT cde_id
  FROM   data_exploration_hub.marketing_hub.GS_CC_Members
),
txn AS (
  SELECT *
  FROM   cpx_dataanalytics_gold.customer360.c360_f_transaction
  WHERE  VisitDateId > '20210101'
    AND  TicketFLAG  = 'Yes'
)
SELECT m.cde_id,
       TO_DATE(CAST(t.VisitDateId AS STRING), 'yyyyMMdd')           AS VisitDateId,
       t.TransactionDateid,
       t.Transaction_Number,
       sc.sales_channel,
       f.title_name,
       i.Item_Description,
       tt.ticket_type_description_english,
       CASE t.CineClub_Discount_Type_ID
         WHEN 0 THEN 'Non CineClub member'
         WHEN 1 THEN 'CineClub member no discount'
         WHEN 2 THEN 'CineClub member 20% on concession'
         WHEN 3 THEN 'CineClub member 20% on games'
         WHEN 4 THEN 'CineClub member 10% on movie rentals and purchases'
         WHEN 5 THEN 'CineClub member $9 ticket'
         WHEN 6 THEN 'CineClub member free ticket'
       END                                                          AS Discount_Type,
       t.Quantity,
       t.SCENE_Points_Earned,
       t.SCENE_Points_Redeemed,
       t.Net_Spend,
       t.Net_Revenue,
       t.AdvanceFLAG,
       t.TicketFLAG,
       t.ConcessionFLAG,
       loc.Location_Name,
       loc.Province_CD
FROM   members m
LEFT  JOIN txn t
        ON m.cde_id = t.CDE_ID
INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i
        ON t.ItemSkey = i.itemskey
INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_film f
        ON t.film_id = f.film_id
INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc
        ON t.LocationId = loc.LocationID
INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_ticket_type tt
        ON t.TicketTypeSkey = tt.tickettypeskey
INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_sales_channel sc
        ON t.SalesChannelSkey = sc.saleschannelskey;